In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sparrow as sp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json

from napari_spatialdata import Interactive

from sparrow.io._visium_hd import visium_hd

from spatialdata.transformations import get_transformation
from sparrow.image._image import _get_spatial_element

import scanpy as sc
from sparrow.table import add_table_layer
from sparrow.points import add_points_layer
from sparrow.shape import add_shapes_layer
from sparrow.utils._keys import _INSTANCE_KEY, _REGION_KEY

import dask.dataframe as dd

from skimage import io

the value of the environment variable BASIC_DCT_BACKEND is not in ["JAX","SCIPY"]
2024-11-08 15:13:00,455 - sparrow.image.pixel_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.im.flowsom', please install this library.
2024-11-08 15:13:01,953 - sparrow.table.cell_clustering._clustering - WARNING - 'flowsom' not installed, to use 'sp.tb.flowsom', please install this library.
2024-11-08 15:13:02,311 - sparrow.plot._qc_cells - WARNING - 'joypy' not installed, to use 'sp.pl.ridgeplot_channel' and 'sp.pl.ridgeplot_channel_sample', please install this library.
2024-11-08 15:13:02,314 - sparrow.plot._qc_image - WARNING - 'textalloc' not installed, to use 'sp.pl.group_snr_ratio' and 'sp.pl.snr_ratio', please install this library.


## Create sdata

In [4]:
sdata = visium_hd(
    path=r'e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs', 
    bin_size=[2], 
    dataset_id="", # Had to adapt spatialdata io visium reader according to https://github.com/scverse/spatialdata-io/issues/218
    bins_as_squares=True, 
    fullres_image_file=r'e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\IMAGES\SCA003_HE_hires.tif', # Adapted harpy visium hd wrapper to take this argument, I also had to set scale_factors=None in line 361
    output= None) # I run into buffer issues if I try to write the sdata to disk
# I set filtered_counts_file=False in harpy visium hd wrapper

# se=_get_spatial_element(sdata, layer="_full_image")

# arr=se.data.rechunk("auto")
# arr_reshaped = arr.transpose(1, 2, 0)

# sdata=sp.im.add_image_layer(sdata, arr=arr_reshaped, transformations=get_transformation(sdata[ "_full_image" ], get_all=True), output_layer="_full_image", overwrite=True, scale_factors=(2, 2, 2, 2))
# sdata.write(r'E:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\1\SCA002_reseq\registration_tests\harpy\sdata.zarr')

sdata

c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<tifffile.TiffFile 'SCA003_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


SpatialData object
├── Images
│     ├── '_full_image': DataArray[cyx] (3, 15241, 16021)
│     ├── '_hires_image': DataArray[cyx] (3, 5708, 6000)
│     └── '_lowres_image': DataArray[cyx] (3, 571, 600)
├── Labels
│     └── '_square_002um_labels': DataArray[yx] (3350, 3350)
├── Shapes
│     └── '_square_002um': GeoDataFrame shape: (7512651, 1) (2D shapes)
└── Tables
      └── 'square_002um': AnnData (7512651, 32285)
with coordinate systems:
    ▸ 'downscaled_hires', with elements:
        _hires_image (Images), _square_002um_labels (Labels), _square_002um (Shapes)
    ▸ 'downscaled_lowres', with elements:
        _lowres_image (Images), _square_002um_labels (Labels), _square_002um (Shapes)
    ▸ 'global', with elements:
        _full_image (Images), _square_002um_labels (Labels), _square_002um (Shapes)

In [5]:
probe_set_df = pd.read_csv(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['square_002um'][:, sdata.tables['square_002um'].var['gene_ids'].isin(included)]

In [6]:
print(f"Number of spots in tissue: {(adata.obs['in_tissue'] == 1).sum()}")

Number of spots in tissue: 6266170


In [7]:
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

Total number of genes: 19011


In [8]:
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 19011


In [9]:
print(f"Total number of UMIs: {adata.X.sum()}")

Total number of UMIs: 216155184.0


In [10]:
print(f"Number of UMIs in tissue: {adata[adata.obs['in_tissue'] == 1].X.sum()}")

Number of UMIs in tissue: 213456288.0


In [11]:
included

{'ENSMUSG00000052504',
 'ENSMUSG00000031605',
 'ENSMUSG00000005360',
 'ENSMUSG00000057609',
 'ENSMUSG00000070985',
 'ENSMUSG00000026353',
 'ENSMUSG00000110519',
 'ENSMUSG00000022800',
 'ENSMUSG00000020037',
 'ENSMUSG00000000182',
 'ENSMUSG00000084989',
 'ENSMUSG00000018634',
 'ENSMUSG00000028223',
 'ENSMUSG00000061689',
 'ENSMUSG00000020832',
 'ENSMUSG00000003948',
 'ENSMUSG00000042831',
 'ENSMUSG00000027963',
 'ENSMUSG00000066586',
 'ENSMUSG00000025580',
 'ENSMUSG00000042489',
 'ENSMUSG00000054733',
 'ENSMUSG00000027706',
 'ENSMUSG00000055415',
 'ENSMUSG00000063065',
 'ENSMUSG00000045030',
 'ENSMUSG00000056130',
 'ENSMUSG00000038750',
 'ENSMUSG00000025007',
 'ENSMUSG00000053754',
 'ENSMUSG00000007655',
 'ENSMUSG00000035578',
 'ENSMUSG00000003378',
 'ENSMUSG00000006191',
 'ENSMUSG00000015405',
 'ENSMUSG00000015605',
 'ENSMUSG00000068574',
 'ENSMUSG00000036962',
 'ENSMUSG00000027329',
 'ENSMUSG00000015880',
 'ENSMUSG00000024197',
 'ENSMUSG00000040824',
 'ENSMUSG00000034731',
 'ENSMUSG00

In [15]:
import h5py

# Open the HDF5 file
with h5py.File(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\molecule_info.h5", "r") as f:
    probe_idx = f['/probes/probe_id']
    probe_id = f['/probes/probe_id']
    count = f["/count"]

    total_sum = count[:].sum()
    total_count = count.size

    print("Sum of all numbers:", total_sum)
    print("Total number of entries:", total_count)


Sum of all numbers: 543649612
Total number of entries: 216424303


In [ ]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


Total number of UMIs for included genes: 542977978
Number of unique UMIs for included genes: 216154979


In [55]:
np.where(included_feature_mask)

(array([    0,     4,     5, ..., 32206, 32207, 32231], dtype=int64),)

In [53]:
np.isin(feature_id, list(included))

array([ True, False, False, ..., False, False, False])

In [49]:
1-(total_count/total_sum)

0.6019083871574622